# Output Visualization Notebook

This notebook reads already-generated CSV files from `outputs/` and creates comparison tables and plots. It does not call Neuronpedia.

It supports the current competence output files and older French `_fr.csv` expertise-style files by normalizing them into one shared table.


## 1. Setup


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display


if Path.cwd().name == "profession_concept_overlap":
    BASE_DIR = Path.cwd()
else:
    BASE_DIR = Path("profession_concept_overlap")

OUTPUT_DIR = BASE_DIR / "outputs"
EPSILON = 1e-9

GENDER_COLORS = {"control": "#b8b8b8", "male": "#8ecae6", "female": "#f4a6c1"}
DIVERGING_CMAP = sns.diverging_palette(330, 220, s=75, l=78, as_cmap=True)
LANGUAGE_ORDER = ["english", "traditional_chinese", "french"]
GENDER_ORDER = ["control", "male", "female"]

sns.set_theme(style="whitegrid", context="notebook", palette="pastel")
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 200)

OUTPUT_DIR


## 2. Load Output Files


In [ ]:
def read_csv_if_exists(path: Path) -> pd.DataFrame:
    if path.exists() and path.stat().st_size > 0:
        return pd.read_csv(path)
    return pd.DataFrame()


def add_missing_language(frame: pd.DataFrame, language: str) -> pd.DataFrame:
    if frame.empty:
        return frame
    frame = frame.copy()
    if "language" not in frame.columns:
        frame["language"] = language
    elif frame["language"].isna().any():
        frame["language"] = frame["language"].fillna(language)
    return frame


trait_records = read_csv_if_exists(OUTPUT_DIR / "profession_gender_trait_fixed_feature_records.csv")
trait_group_summary = read_csv_if_exists(OUTPUT_DIR / "profession_gender_trait_group_summary.csv")
trait_feature_summary = read_csv_if_exists(OUTPUT_DIR / "profession_gender_trait_feature_summary_long.csv")

french_records = add_missing_language(
    read_csv_if_exists(OUTPUT_DIR / "profession_gender_expertise_fixed_feature_records_fr.csv"),
    "french",
)
french_group_summary = add_missing_language(
    read_csv_if_exists(OUTPUT_DIR / "profession_gender_expertise_group_summary_fr.csv"),
    "french",
)
french_feature_summary = add_missing_language(
    read_csv_if_exists(OUTPUT_DIR / "profession_gender_expertise_feature_summary_long_fr.csv"),
    "french",
)

for frame in [trait_records, trait_group_summary, trait_feature_summary, french_records, french_group_summary, french_feature_summary]:
    if not frame.empty and "trait_category" not in frame.columns:
        frame["trait_category"] = "competence"

print("trait_records", trait_records.shape)
print("trait_group_summary", trait_group_summary.shape)
print("french_records", french_records.shape)
print("french_group_summary", french_group_summary.shape)


## 3. Normalize And Combine


In [ ]:
def normalize_records(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    needed = [
        "language",
        "trait_category",
        "profession",
        "gender_group",
        "prompt",
        "prompt_pair",
        "feature",
        "source",
        "feature_index",
        "attribute_activation",
        "content_activation",
        "role_activation",
        "max_activation",
    ]
    normalized = frame.copy()
    for column in needed:
        if column not in normalized.columns:
            normalized[column] = pd.NA
    return normalized[needed]


def normalize_group_summary(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    needed = [
        "language",
        "trait_category",
        "profession",
        "gender_group",
        "prompt_count",
        "tested_feature_count",
        "mean_attribute_activation",
        "mean_content_activation",
        "mean_role_activation",
    ]
    normalized = frame.copy()
    for column in needed:
        if column not in normalized.columns:
            normalized[column] = pd.NA
    return normalized[needed]


activation_records = pd.concat(
    [normalize_records(trait_records), normalize_records(french_records)],
    ignore_index=True,
).dropna(subset=["language", "profession", "gender_group"], how="any")

group_summary = pd.concat(
    [normalize_group_summary(trait_group_summary), normalize_group_summary(french_group_summary)],
    ignore_index=True,
).dropna(subset=["language", "profession", "gender_group"], how="any")

# If a summary CSV is missing, derive group_summary from raw activation records.
if group_summary.empty and not activation_records.empty:
    group_summary = (
        activation_records
        .groupby(["language", "trait_category", "profession", "gender_group"], as_index=False)
        .agg(
            prompt_count=("prompt", "nunique"),
            tested_feature_count=("feature", "nunique"),
            mean_attribute_activation=("attribute_activation", "mean"),
            mean_content_activation=("content_activation", "mean"),
            mean_role_activation=("role_activation", "mean"),
        )
    )

for column in ["mean_attribute_activation", "mean_content_activation", "mean_role_activation"]:
    if column in group_summary.columns:
        group_summary[column] = pd.to_numeric(group_summary[column], errors="coerce")

group_summary["language"] = pd.Categorical(group_summary["language"], categories=LANGUAGE_ORDER, ordered=True)
group_summary["gender_group"] = pd.Categorical(group_summary["gender_group"], categories=GENDER_ORDER, ordered=True)
group_summary = group_summary.sort_values(["language", "profession", "gender_group"]).reset_index(drop=True)

display(group_summary.head(20))
print("languages:", list(group_summary["language"].dropna().unique()))
print("professions:", sorted(group_summary["profession"].dropna().unique()))


## 4. Difference Tables


In [ ]:
def pct_relative_to_control(value: pd.Series, control: pd.Series) -> pd.Series:
    denominator = control.where(control.abs() > EPSILON)
    return ((value - control) / denominator) * 100


def signed_pct_difference(left: pd.Series, right: pd.Series) -> pd.Series:
    denominator = ((left.abs() + right.abs()) / 2).where(lambda s: s > EPSILON)
    return ((left - right) / denominator) * 100


group_difference_table = (
    group_summary
    .pivot_table(
        index=["language", "trait_category", "profession"],
        columns="gender_group",
        values="mean_attribute_activation",
        fill_value=0.0,
        observed=False,
    )
    .reset_index()
)

for column in ["control", "male", "female"]:
    if column not in group_difference_table.columns:
        group_difference_table[column] = 0.0

group_difference_table["male_minus_control"] = group_difference_table["male"] - group_difference_table["control"]
group_difference_table["female_minus_control"] = group_difference_table["female"] - group_difference_table["control"]
group_difference_table["male_minus_female"] = group_difference_table["male"] - group_difference_table["female"]
group_difference_table["male_pct_vs_control"] = pct_relative_to_control(group_difference_table["male"], group_difference_table["control"])
group_difference_table["female_pct_vs_control"] = pct_relative_to_control(group_difference_table["female"], group_difference_table["control"])
group_difference_table["male_female_pct_gap"] = signed_pct_difference(group_difference_table["male"], group_difference_table["female"])

PROFESSION_CANONICAL_MAP = {
    "doctor": "doctor",
    "médecin": "doctor",
    "engineer": "engineer",
    "pilot": "pilot",
    "pilote": "pilot",
    "teacher": "teacher",
    "professeur": "teacher",
    "spécialiste": "specialist",
}

LANGUAGE_LABELS = {
    "english": "english",
    "traditional_chinese": "chinese",
    "french": "french",
}

male_female_comparison_table = group_difference_table[
    ["language", "profession", "male", "female", "male_minus_female", "male_female_pct_gap"]
].copy()
male_female_comparison_table["language_label"] = male_female_comparison_table["language"].map(LANGUAGE_LABELS).fillna(male_female_comparison_table["language"].astype(str))
male_female_comparison_table["occupation"] = male_female_comparison_table["profession"].map(PROFESSION_CANONICAL_MAP).fillna(male_female_comparison_table["profession"].astype(str))
male_female_comparison_table["higher_group"] = "tie"
male_female_comparison_table.loc[male_female_comparison_table["male_female_pct_gap"] > 0, "higher_group"] = "male"
male_female_comparison_table.loc[male_female_comparison_table["male_female_pct_gap"] < 0, "higher_group"] = "female"
male_female_comparison_table = male_female_comparison_table[
    ["language", "language_label", "occupation", "profession", "higher_group", "male", "female", "male_minus_female", "male_female_pct_gap"]
].sort_values(["occupation", "language_label", "profession"])

display(group_difference_table)
display(male_female_comparison_table)


## 5. Styled Male-Female Comparison Table


In [ ]:
def format_signed_pct(value: float) -> str:
    if pd.isna(value):
        return ""
    return f"{value:+.1f}%"


def blend_rgb(start: tuple[int, int, int], end: tuple[int, int, int], weight: float) -> str:
    weight = max(0.0, min(1.0, float(weight)))
    rgb = [round(start_value + (end_value - start_value) * weight) for start_value, end_value in zip(start, end)]
    return f"rgb({rgb[0]}, {rgb[1]}, {rgb[2]})"


def style_signed_percentage_table(frame: pd.DataFrame, pct_columns: list[str]):
    max_abs = frame[pct_columns].abs().max().max()
    max_abs = float(max_abs) if pd.notna(max_abs) and max_abs > EPSILON else 1.0
    white = (255, 255, 255)
    male_blue = (142, 202, 230)
    female_pink = (244, 166, 193)

    def color_signed_pct(value):
        if pd.isna(value):
            return ""
        weight = min(abs(float(value)) / max_abs, 1.0)
        if value > 0:
            background = blend_rgb(white, male_blue, weight)
        elif value < 0:
            background = blend_rgb(white, female_pink, weight)
        else:
            background = "white"
        return f"background-color: {background}; color: #111111; font-weight: 600"

    return (
        frame.style
        .format({column: format_signed_pct for column in pct_columns})
        .format({"male": "{:.3f}", "female": "{:.3f}", "male_minus_female": "{:+.3f}"})
        .map(color_signed_pct, subset=pct_columns)
        .set_properties(subset=pct_columns, **{"text-align": "center"})
    )


language_pct_columns = ["english", "chinese", "french"]
occupation_language_pct_table = (
    male_female_comparison_table
    .pivot_table(
        index="occupation",
        columns="language_label",
        values="male_female_pct_gap",
        aggfunc="mean",
        observed=False,
    )
    .reindex(columns=language_pct_columns)
    .reset_index()
)

occupation_language_winner_table = (
    male_female_comparison_table
    .pivot_table(
        index="occupation",
        columns="language_label",
        values="higher_group",
        aggfunc=lambda values: ", ".join(sorted(set(str(value) for value in values))),
        observed=False,
    )
    .reindex(columns=language_pct_columns)
    .reset_index()
)

occupation_language_pct_table.to_csv(OUTPUT_DIR / "occupation_language_male_female_pct_table.csv", index=False)
occupation_language_winner_table.to_csv(OUTPUT_DIR / "occupation_language_male_female_winner_table.csv", index=False)

display(style_signed_percentage_table(occupation_language_pct_table, language_pct_columns))
display(occupation_language_winner_table)
display(style_signed_percentage_table(male_female_comparison_table, ["male_female_pct_gap"]))


## 6. Visualization Gallery


### 1. Mean Activation By Language, Occupation, And Gender


In [ ]:
plot_data = group_summary.dropna(subset=["mean_attribute_activation"]).copy()
grid = sns.catplot(
    data=plot_data,
    x="profession",
    y="mean_attribute_activation",
    hue="gender_group",
    col="language",
    kind="bar",
    hue_order=GENDER_ORDER,
    palette=GENDER_COLORS,
    height=4,
    aspect=1.15,
)
grid.set_axis_labels("", "mean attribute-token activation")
grid.set_titles("{col_name}")
for ax in grid.axes.flat:
    ax.tick_params(axis="x", rotation=20)
plt.show()


### 2. Signed Male-Female Percentage Gap


In [ ]:
plot_data = male_female_comparison_table.copy()
plot_data["label"] = plot_data["language"].astype(str) + " | " + plot_data["profession"].astype(str)
plot_data = plot_data.sort_values("male_female_pct_gap")
colors = [GENDER_COLORS["male"] if value >= 0 else GENDER_COLORS["female"] for value in plot_data["male_female_pct_gap"]]

fig, ax = plt.subplots(figsize=(9, max(4, 0.45 * len(plot_data))), constrained_layout=True)
ax.barh(plot_data["label"], plot_data["male_female_pct_gap"], color=colors)
ax.axvline(0, color="#333333", linewidth=0.9)
ax.set_title("Male vs female competence activation gap")
ax.set_xlabel("signed % gap: positive = male higher, negative = female higher")
ax.set_ylabel("")
plt.show()


### 3. Language-Occupation Signed Gap Matrix


In [ ]:
matrix = male_female_comparison_table.pivot_table(
    index="language",
    columns="profession",
    values="male_female_pct_gap",
    observed=False,
)
max_abs = max(float(matrix.abs().max().max()), EPSILON)
fig, ax = plt.subplots(figsize=(8.5, max(3.2, 0.7 * len(matrix))), constrained_layout=True)
sns.heatmap(
    matrix,
    annot=True,
    fmt="+.1f",
    cmap=DIVERGING_CMAP,
    center=0,
    vmin=-max_abs,
    vmax=max_abs,
    linewidths=0.8,
    linecolor="white",
    cbar_kws={"label": "signed % gap"},
    ax=ax,
)
ax.set_title("Signed male-female percentage gap by language and occupation")
ax.set_xlabel("")
ax.set_ylabel("")
plt.show()


### 4. Male And Female Change Relative To Control


In [ ]:
control_long = group_difference_table.melt(
    id_vars=["language", "profession"],
    value_vars=["male_pct_vs_control", "female_pct_vs_control"],
    var_name="comparison",
    value_name="pct_vs_control",
).dropna(subset=["pct_vs_control"])
control_long["gender"] = control_long["comparison"].str.replace("_pct_vs_control", "", regex=False)

grid = sns.catplot(
    data=control_long,
    x="profession",
    y="pct_vs_control",
    hue="gender",
    col="language",
    kind="bar",
    palette={"male": GENDER_COLORS["male"], "female": GENDER_COLORS["female"]},
    height=4,
    aspect=1.15,
)
for ax in grid.axes.flat:
    ax.axhline(0, color="#333333", linewidth=0.9)
    ax.tick_params(axis="x", rotation=20)
grid.set_axis_labels("", "% change relative to control")
grid.set_titles("{col_name}")
plt.show()


### 5. Male Versus Female Scatter


In [ ]:
plot_data = male_female_comparison_table.copy()
fig, ax = plt.subplots(figsize=(6.5, 6), constrained_layout=True)
sns.scatterplot(
    data=plot_data,
    x="female",
    y="male",
    hue="language",
    style="profession",
    s=110,
    ax=ax,
)
limit_min = min(plot_data["female"].min(), plot_data["male"].min())
limit_max = max(plot_data["female"].max(), plot_data["male"].max())
ax.plot([limit_min, limit_max], [limit_min, limit_max], color="#333333", linestyle="--", linewidth=1)
ax.set_title("Male vs female mean activation")
ax.set_xlabel("female mean activation")
ax.set_ylabel("male mean activation")
plt.show()


### 6. Activation Distribution By Gender


In [ ]:
if activation_records.empty:
    print("No raw activation records available.")
else:
    distribution_data = activation_records.dropna(subset=["attribute_activation"]).copy()
    distribution_data["gender_group"] = pd.Categorical(distribution_data["gender_group"], categories=GENDER_ORDER, ordered=True)
    grid = sns.catplot(
        data=distribution_data,
        x="gender_group",
        y="attribute_activation",
        hue="gender_group",
        col="language",
        kind="boxen",
        palette=GENDER_COLORS,
        height=4,
        aspect=1.05,
        legend=False,
    )
    grid.set_axis_labels("", "prompt-feature activation")
    grid.set_titles("{col_name}")
    plt.show()


### 7. Occupation-Level Gender Gap Ranking By Language


In [ ]:
plot_data = male_female_comparison_table.sort_values(["language", "male_female_pct_gap"]).copy()
grid = sns.FacetGrid(plot_data, col="language", height=4, aspect=1.1, sharex=False)

def draw_ranked_bars(data, **kwargs):
    colors = [GENDER_COLORS["male"] if value >= 0 else GENDER_COLORS["female"] for value in data["male_female_pct_gap"]]
    plt.barh(data["profession"], data["male_female_pct_gap"], color=colors)
    plt.axvline(0, color="#333333", linewidth=0.9)

grid.map_dataframe(draw_ranked_bars)
grid.set_axis_labels("signed % gap", "")
grid.set_titles("{col_name}")
plt.show()


### 8. Feature-Level Top Gender Gaps


In [ ]:
if activation_records.empty:
    print("No raw activation records available.")
else:
    feature_summary = (
        activation_records
        .groupby(["language", "profession", "feature", "gender_group"], as_index=False)
        .agg(mean_attribute_activation=("attribute_activation", "mean"))
    )
    feature_diff = (
        feature_summary
        .pivot_table(
            index=["language", "profession", "feature"],
            columns="gender_group",
            values="mean_attribute_activation",
            fill_value=0.0,
            observed=False,
        )
        .reset_index()
    )
    for column in ["male", "female"]:
        if column not in feature_diff.columns:
            feature_diff[column] = 0.0
    feature_diff["male_minus_female"] = feature_diff["male"] - feature_diff["female"]
    feature_diff["abs_gap"] = feature_diff["male_minus_female"].abs()

    top_features = feature_diff.sort_values("abs_gap", ascending=False).head(20).copy()
    top_features["label"] = top_features["language"].astype(str) + " | " + top_features["profession"].astype(str) + " | " + top_features["feature"].astype(str)
    colors = [GENDER_COLORS["male"] if value >= 0 else GENDER_COLORS["female"] for value in top_features["male_minus_female"]]

    fig, ax = plt.subplots(figsize=(10, max(5, 0.38 * len(top_features))), constrained_layout=True)
    ax.barh(top_features["label"], top_features["male_minus_female"], color=colors)
    ax.axvline(0, color="#333333", linewidth=0.9)
    ax.invert_yaxis()
    ax.set_title("Top feature-level male-female activation gaps")
    ax.set_xlabel("male minus female mean activation")
    ax.set_ylabel("")
    plt.show()


### 9. Prompt-Level Matched Male-Female Gaps


In [ ]:
if activation_records.empty or "prompt_pair" not in activation_records.columns:
    print("No prompt-pair activation records available.")
else:
    prompt_summary = (
        activation_records[activation_records["gender_group"].isin(["male", "female"])]
        .groupby(["language", "profession", "prompt_pair", "gender_group"], as_index=False)
        .agg(mean_attribute_activation=("attribute_activation", "mean"))
    )
    prompt_diff = (
        prompt_summary
        .pivot_table(
            index=["language", "profession", "prompt_pair"],
            columns="gender_group",
            values="mean_attribute_activation",
            fill_value=0.0,
            observed=False,
        )
        .reset_index()
    )
    for column in ["male", "female"]:
        if column not in prompt_diff.columns:
            prompt_diff[column] = 0.0
    prompt_diff["male_minus_female"] = prompt_diff["male"] - prompt_diff["female"]
    prompt_diff["label"] = prompt_diff["language"].astype(str) + " | " + prompt_diff["profession"].astype(str) + " | " + prompt_diff["prompt_pair"].astype(str)
    prompt_diff = prompt_diff.sort_values("male_minus_female")
    colors = [GENDER_COLORS["male"] if value >= 0 else GENDER_COLORS["female"] for value in prompt_diff["male_minus_female"]]

    fig, ax = plt.subplots(figsize=(10, max(5, 0.26 * len(prompt_diff))), constrained_layout=True)
    ax.barh(prompt_diff["label"], prompt_diff["male_minus_female"], color=colors)
    ax.axvline(0, color="#333333", linewidth=0.9)
    ax.set_title("Prompt-level male-female activation gaps")
    ax.set_xlabel("male minus female mean activation")
    ax.set_ylabel("")
    plt.show()


### 10. Data Coverage By Language And Occupation


In [ ]:
if activation_records.empty:
    print("No raw activation records available.")
else:
    coverage = (
        activation_records
        .groupby(["language", "profession", "gender_group"], as_index=False)
        .agg(record_count=("attribute_activation", "size"))
    )
    grid = sns.catplot(
        data=coverage,
        x="profession",
        y="record_count",
        hue="gender_group",
        col="language",
        kind="bar",
        hue_order=GENDER_ORDER,
        palette=GENDER_COLORS,
        height=4,
        aspect=1.15,
    )
    grid.set_axis_labels("", "activation record count")
    grid.set_titles("{col_name}")
    for ax in grid.axes.flat:
        ax.tick_params(axis="x", rotation=20)
    plt.show()
